# EDA & Data Preprocessing — Production-Grade Data Quality Engineering

**Phase:** Data & Systems Foundations  
**Prerequisites:** NB01 (Python Primer), NB03 (Cloud Platforms)  
**Toolchain:** Pandas · NumPy · Seaborn · scikit-learn · ydata-profiling  
**Objective:** Master the complete data quality pipeline — from raw ingestion through cleaned, validated, ML-ready datasets. Learn to diagnose missingness mechanisms, detect multicollinearity, handle class imbalance, and build reproducible preprocessing pipelines.

---

## Why This Notebook Is Non-Negotiable

**80% of ML engineering time is spent on data.** Not on fancy architectures. Not on hyperparameter tuning. On understanding, cleaning, validating, and transforming raw data into something a model can learn from.

A model trained on dirty data is a liability:
- One undetected outlier can shift a regression coefficient by 300%
- One leaky feature turns 95% accuracy into 50% production accuracy
- One miscoded categorical variable introduces silent bias

This notebook teaches you to find and fix these problems **before** they reach a model.

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat EDA as "make some plots." Seniors use EDA to discover data quality issues BEFORE they corrupt a model. One missing-value strategy can shift model accuracy by 5%+. One leaky feature can make your model useless in production.

**Mental Model:** EDA is a medical exam for your data. You're checking vital signs (distributions), looking for symptoms (outliers, nulls), and diagnosing conditions (data leakage, class imbalance) before surgery (model training).

**Common Junior Pitfall:** Training on data with target leakage — a feature that "magically" contains information from the future. Example: including `account_closed_date` to predict churn. The model gets 99% accuracy in dev, 50% in production.

---


## 📑 Table of Contents

* [Why This Notebook Is Non-Negotiable](#why-this-notebook-is-non-negotiable)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Pandas Mastery: The Operations You'll Use Daily](#1--pandas-mastery-the-operations-youll-use-daily)
* [2 · Visual EDA: Finding Problems Your Eyes Catch Faster Than Code](#2--visual-eda-finding-problems-your-eyes-catch-faster-than-code)
* [3 · Missingness Analysis: MCAR, MAR & MNAR](#3--missingness-analysis-mcar-mar--mnar)
* [4 · Data Cleaning Pipeline](#4--data-cleaning-pipeline)
* [5 · Feature Engineering for ML](#5--feature-engineering-for-ml)
  * [5.1 Encoding Strategies](#51-encoding-strategies)
  * [5.2 Scaling & Transformation](#52-scaling--transformation)
* [6 · Multicollinearity & Feature Diagnostics](#6--multicollinearity--feature-diagnostics)
* [7 · Class Imbalance Handling](#7--class-imbalance-handling)
* [8 · Production Pipeline with ColumnTransformer](#8--production-pipeline-with-columntransformer)
* [9 · Automated EDA Tools](#9--automated-eda-tools)
* [10 · Data Validation (Connects to NB07)](#10--data-validation-connects-to-nb07)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


In [ ]:
# Cell 0 — Install dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels


---
## 1 · Pandas Mastery: The Operations You'll Use Daily

Before any analysis, you need fluency in the 5 diagnostic commands that every senior engineer runs on a new dataset within the first 60 seconds.


In [ ]:
# Cell 1 — Create realistic messy dataset
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

data = pd.DataFrame({
    'customer_id': range(1000, 1000+n),
    'age': np.random.normal(35, 12, n).astype(int),
    'income': np.random.lognormal(10.5, 0.8, n).round(2),
    'credit_score': np.random.normal(680, 80, n).astype(int),
    'account_age_months': np.random.exponential(36, n).astype(int),
    'num_products': np.random.poisson(2.5, n),
    'has_credit_card': np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'is_active': np.random.choice([0, 1], n, p=[0.2, 0.8]),
    'country': np.random.choice(['US', 'UK', 'DE', 'FR', 'JP', 'BR'], n),
    'satisfaction_score': np.random.choice([1,2,3,4,5], n, p=[0.05,0.1,0.2,0.35,0.3]),
    'churned': np.random.choice([0, 1], n, p=[0.8, 0.2]),
})

# Inject REALISTIC data problems
# 1. Missing values — DIFFERENT MECHANISMS
# MCAR: random missingness in satisfaction_score
mcar_mask = np.random.random(n) < 0.08
data.loc[mcar_mask, 'satisfaction_score'] = np.nan

# MAR: income is more likely missing for churned customers
mar_mask = (data['churned'] == 1) & (np.random.random(n) < 0.25)
mar_mask |= (data['churned'] == 0) & (np.random.random(n) < 0.05)
data.loc[mar_mask, 'income'] = np.nan

# MNAR: credit_score missing for people with LOW credit scores
mnar_mask = (data['credit_score'] < 600) & (np.random.random(n) < 0.40)
data.loc[mnar_mask, 'credit_score'] = np.nan

# 2. Outliers (data entry errors)
data.loc[42, 'age'] = 350
data.loc[99, 'income'] = -5000
data.loc[150, 'credit_score'] = 9999

# 3. Duplicates
data = pd.concat([data, data.iloc[:5]], ignore_index=True)

# 4. Correlated features (for multicollinearity demo)
data['income_estimate'] = data['income'] * np.random.normal(1.0, 0.05, len(data))

print(f'Dataset: {data.shape[0]} rows × {data.shape[1]} columns')
print(f'\nData types:')
print(data.dtypes.to_string())
print(f'\nMissing values:')
print(data.isnull().sum()[data.isnull().sum() > 0].to_string())


In [ ]:
# Cell 2 — The 5 commands you run FIRST on any dataset
import pandas as pd

# 1. Shape & types
print(f'Shape: {data.shape}')
print(f'\n--- .info() ---')
data.info()

# 2. Statistical summary
print(f'\n--- .describe() ---')
print(data.describe().round(2).to_string())

# 3. Missing values (% is critical — not just count)
print(f'\n--- Missing Values ---')
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)
print(pd.DataFrame({'count': missing[missing>0], 'percent': missing_pct[missing_pct>0]}).to_string())

# 4. Duplicates
print(f'\nDuplicate rows: {data.duplicated().sum()}')

# 5. Target distribution (THE most important check for classification)
print(f'\nTarget distribution (churned):')
print(data['churned'].value_counts(normalize=True).round(3).to_string())
ratio = data['churned'].value_counts()[0] / data['churned'].value_counts()[1]
print(f'Class imbalance ratio: {ratio:.1f}:1')
if ratio > 3:
    print('⚠️  Significant class imbalance detected — see Section 7 for handling strategies')


---
## 2 · Visual EDA: Finding Problems Your Eyes Catch Faster Than Code

A histogram reveals outliers in 0.5 seconds. A correlation heatmap shows multicollinearity instantly. Visual EDA is not optional — it's a senior engineer's primary diagnostic tool.

### The EDA Dashboard Pattern

Production data scientists build **reusable dashboards** — not one-off plots. The 6-panel layout below covers the critical diagnostics:

| Panel | What It Reveals | Action if Abnormal |
|-------|----------------|--------------------|
| Distribution | Outliers, skewness | Cap/remove outliers, log-transform |
| Correlation heatmap | Multicollinearity | Drop or combine correlated features |
| Target distribution | Class imbalance | SMOTE, class weights, or stratified splitting |
| Feature vs target | Predictive power | Keep strong features, investigate weak ones |
| Missing pattern | Systematic vs random | Choose MCAR/MAR/MNAR strategy |
| Box plots | Outlier magnitude | IQR filtering or domain-based capping |


In [ ]:
# Cell 3 — Visual EDA Dashboard
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA Dashboard: Churn Prediction Dataset', fontsize=14, fontweight='bold')

# 1. Age distribution (spot outlier: age=350)
axes[0,0].hist(data['age'].dropna(), bins=40, color='steelblue', alpha=0.8)
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].annotate('Outlier!', xy=(350, 1), fontsize=10, color='red', fontweight='bold')

# 2. Income distribution (log-normal)
axes[0,1].hist(data['income'].dropna(), bins=40, color='coral', alpha=0.8)
axes[0,1].set_title('Income Distribution (log-normal)')

# 3. Target distribution
data['churned'].value_counts().plot.bar(ax=axes[0,2], color=['steelblue', 'coral'])
axes[0,2].set_title('Target: Churned (Class Imbalance!)')
axes[0,2].set_xticklabels(['Not Churned', 'Churned'], rotation=0)

# 4. Correlation heatmap
numeric_cols = data.select_dtypes(include=[np.number]).columns[:7]
sns.heatmap(data[numeric_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', ax=axes[1,0],
            center=0, vmin=-1, vmax=1)
axes[1,0].set_title('Feature Correlations')

# 5. Feature vs target (churn by country)
data.groupby('country')['churned'].mean().sort_values().plot.barh(ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Churn Rate by Country')
axes[1,1].set_xlabel('Churn Rate')

# 6. Box plots for outlier detection
data[['age', 'credit_score']].boxplot(ax=axes[1,2])
axes[1,2].set_title('Box Plots (Outlier Detection)')

plt.tight_layout()
plt.show()

print('Key Findings from Visual EDA:')
print('  1. Age has an outlier at 350 (impossible — data entry error)')
print('  2. Income is log-normal → may need log transform for linear models')
print('  3. Target is imbalanced (80/20) → need stratified splits + class weights')
print('  4. income and income_estimate are highly correlated → multicollinearity risk')
print('  5. income and credit_score have structured missing values → NOT random')


---
## 3 · Missingness Analysis: MCAR, MAR & MNAR

**This is where juniors and seniors diverge.** Juniors call `fillna(df.mean())` and move on. Seniors first diagnose *why* data is missing, because the mechanism determines the correct treatment.

### The Three Missingness Mechanisms

| Mechanism | Definition | Example | Treatment |
|-----------|-----------|---------|----------|
| **MCAR** (Missing Completely At Random) | Missingness is unrelated to any variable | Survey response randomly lost | Safe to drop or impute with mean/median |
| **MAR** (Missing At Random) | Missingness depends on OTHER observed variables | Income missing more for churned users | Impute using related variables (KNN, model-based) |
| **MNAR** (Missing Not At Random) | Missingness depends on the MISSING VALUE itself | Low credit scores are more likely unreported | Most dangerous — needs domain knowledge or indicator variables |

### 🎓 Why This Matters

- **MCAR**: Mean/median imputation is unbiased. Safe and simple.
- **MAR**: Mean imputation is **biased**. You need conditional imputation (e.g., impute income separately for churned vs. non-churned).
- **MNAR**: ALL imputation methods are biased. The best strategy is often to add a `feature_is_missing` indicator column and let the model learn the pattern.

### Diagnostic: How to Detect the Mechanism


In [ ]:
# Cell 4 — Missingness mechanism diagnosis
import pandas as pd
import numpy as np

print('=== Missingness Mechanism Diagnosis ===')
print()

# --- Test 1: MCAR check for satisfaction_score ---
# If MCAR, the churn rate should be EQUAL for rows with/without missing satisfaction
has_sat = data[data['satisfaction_score'].notna()]['churned'].mean()
no_sat = data[data['satisfaction_score'].isna()]['churned'].mean()
print('Test 1: satisfaction_score (should be MCAR)')
print(f'  Churn rate when satisfaction present:  {has_sat:.3f}')
print(f'  Churn rate when satisfaction missing:  {no_sat:.3f}')
print(f'  Difference: {abs(has_sat - no_sat):.3f} → {"✅ MCAR likely (small diff)" if abs(has_sat - no_sat) < 0.05 else "⚠️  MAR/MNAR suspected"}')
print()

# --- Test 2: MAR check for income ---
# If MAR, missingness depends on OTHER variables (like churn status)
has_inc = data[data['income'].notna()]['churned'].mean()
no_inc = data[data['income'].isna()]['churned'].mean()
print('Test 2: income (should be MAR — depends on churn)')
print(f'  Churn rate when income present: {has_inc:.3f}')
print(f'  Churn rate when income missing: {no_inc:.3f}')
print(f'  Difference: {abs(has_inc - no_inc):.3f} → {"⚠️  MAR confirmed (large diff!)" if abs(has_inc - no_inc) > 0.05 else "✅ Looks MCAR"}')
print()

# --- Test 3: MNAR check for credit_score ---
# If MNAR, the OBSERVED credit scores should be biased (higher than true population)
observed_mean = data['credit_score'].dropna().mean()
print('Test 3: credit_score (should be MNAR — low scores are missing)')
print(f'  Observed mean credit score (after dropping NaN): {observed_mean:.1f}')
print(f'  True population mean (before injection): ~680')
print(f'  Bias: +{observed_mean - 680:.1f} points → {"⚠️  MNAR suspected (observed mean inflated!)" if observed_mean > 690 else "Inconclusive"}')
print()

# --- Production Pattern: Add missingness indicators ---
print('Production Strategy for MNAR:')
print('  Add binary indicator columns: credit_score_is_missing = 1 if NaN')
print('  This lets the model learn that "missingness itself is informative"')
data['credit_score_missing'] = data['credit_score'].isna().astype(int)
data['income_missing'] = data['income'].isna().astype(int)
print(f'  Added: credit_score_missing (sum={data["credit_score_missing"].sum()})')
print(f'  Added: income_missing (sum={data["income_missing"].sum()})')


---
## 4 · Data Cleaning Pipeline

Cleaning must happen in a **deterministic order**. The sequence below prevents cascading errors:

```
1. Remove duplicates       → Prevents inflated statistics
2. Fix impossible values   → Prevents outliers from corrupting imputation
3. Handle missing values   → Uses clean statistics for imputation
4. Type conversions        → Ensures downstream compatibility
```

### ⚠️ Critical Rule: Never Impute Before Removing Outliers

If you impute with the mean *before* removing `age=350`, the mean is corrupted. All imputed values inherit the error.


In [ ]:
# Cell 5 — Production data cleaning pipeline
import pandas as pd
import numpy as np

def clean_dataset(df):
    """Production-grade data cleaning pipeline.
    
    Order matters! Clean in this sequence:
    1. Remove duplicates
    2. Fix outliers / impossible values
    3. Handle missing values (mechanism-aware)
    4. Type conversions
    """
    df = df.copy()
    log = []
    
    # 1. Remove duplicates
    n_before = len(df)
    df = df.drop_duplicates()
    log.append(f'Removed {n_before - len(df)} duplicate rows')
    
    # 2. Fix outliers / impossible values
    invalid_age = (df['age'] < 0) | (df['age'] > 120)
    df.loc[invalid_age, 'age'] = np.nan
    log.append(f'Fixed {invalid_age.sum()} impossible age values')
    
    invalid_income = df['income'] < 0
    df.loc[invalid_income, 'income'] = np.nan
    log.append(f'Fixed {invalid_income.sum()} negative income values')
    
    invalid_cs = (df['credit_score'] < 300) | (df['credit_score'] > 850)
    df.loc[invalid_cs, 'credit_score'] = np.nan
    log.append(f'Fixed {invalid_cs.sum()} invalid credit scores')
    
    # 3. Handle missing values — MECHANISM-AWARE
    # MCAR (satisfaction_score): safe to use median
    df['satisfaction_score'].fillna(df['satisfaction_score'].median(), inplace=True)
    
    # MAR (income): impute separately by churn status
    for churn_val in [0, 1]:
        mask = (df['churned'] == churn_val) & (df['income'].isna())
        fill_val = df.loc[df['churned'] == churn_val, 'income'].median()
        df.loc[mask, 'income'] = fill_val
    log.append('Imputed income conditionally by churn status (MAR strategy)')
    
    # MNAR (credit_score): use median + keep indicator column
    df['credit_score'].fillna(df['credit_score'].median(), inplace=True)
    log.append('Imputed credit_score with median + kept missing indicator (MNAR strategy)')
    
    # Age: simple median (small count)
    df['age'].fillna(df['age'].median(), inplace=True)
    
    # 4. Type conversions
    df['customer_id'] = df['customer_id'].astype(str)
    df['churned'] = df['churned'].astype(int)
    
    # Report
    print('=== Cleaning Report ===')
    for entry in log:
        print(f'  ✓ {entry}')
    print(f'  Final: {len(df)} rows, {df.isnull().sum().sum()} remaining NaNs')
    
    return df

df_clean = clean_dataset(data)


---
## 5 · Feature Engineering for ML

### 5.1 Encoding Strategies

Different encodings serve different purposes. Choosing wrong can silently break your model:

| Encoding | When to Use | Example | Risk |
|----------|------------|---------|------|
| **One-Hot** | Low cardinality (<10 unique) | Country (6 values) | Dimensionality if high-card |
| **Ordinal** | Natural order exists | Education: High School < Bachelor < Master | Wrong if no order exists |
| **Target** | High cardinality + classification | City (100+ values) | Leakage if not cross-validated |
| **Frequency** | High cardinality, simple | Product ID | Loses semantic meaning |
| **Binary** | Two categories only | Gender, Yes/No | N/A |

### 5.2 Scaling & Transformation

| Method | Formula | Use When | NOT Needed For |
|--------|---------|----------|---------------|
| **StandardScaler** | $(x - \mu) / \sigma$ | Logistic regression, SVM, NNs | Tree-based models |
| **MinMaxScaler** | $(x - min) / (max - min)$ | Neural networks (bounded activations) | If outliers present |
| **RobustScaler** | $(x - median) / IQR$ | When outliers exist | Tree-based models |
| **Log transform** | $\log(1 + x)$ | Right-skewed distributions (income) | Symmetric data |
| **PowerTransformer** | Box-Cox or Yeo-Johnson | Make any distribution Gaussian | Already normal data |


In [ ]:
# Cell 6 — Encoding and scaling strategies
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
import pandas as pd
import numpy as np

df = df_clean.copy()

# === 1. One-Hot Encoding (low cardinality: country has 6 values) ===
country_dummies = pd.get_dummies(df['country'], prefix='country', drop_first=True)
print('One-hot encoding (country):')
print(f'  Before: 1 column (country)  →  After: {country_dummies.shape[1]} columns (drop_first avoids dummy trap)')

# === 2. Target Encoding (high cardinality — simulated) ===
# Each category → mean of the target variable for that category
# CRITICAL: must use TRAINING set only to avoid leakage
target_means = df.groupby('country')['churned'].mean()
df['country_target_enc'] = df['country'].map(target_means)
print(f'\nTarget encoding (country → churn rate):')
for country, rate in target_means.items():
    print(f'  {country}: {rate:.3f}')
print('  ⚠️  Must compute on TRAIN set only, apply to TEST — otherwise leakage!')

# === 3. Frequency Encoding ===
freq = df['country'].value_counts(normalize=True)
df['country_freq_enc'] = df['country'].map(freq)
print(f'\nFrequency encoding: maps each category to its proportion')

# === 4. Scaling ===
numeric_features = ['age', 'income', 'credit_score', 'account_age_months', 'num_products']
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df[numeric_features]),
    columns=[f'{c}_scaled' for c in numeric_features],
    index=df.index
)
print(f'\nStandardScaler applied:')
print(f'  Before: income range [{df["income"].min():.0f}, {df["income"].max():.0f}]')
print(f'  After:  income range [{df_scaled["income_scaled"].min():.2f}, {df_scaled["income_scaled"].max():.2f}]')

# === 5. Log Transform for Skewed Distributions ===
df['income_log'] = np.log1p(df['income'])
print(f'\nLog transform (income):')
print(f'  Skewness before: {df["income"].skew():.2f}')
print(f'  Skewness after:  {df["income_log"].skew():.2f}  (closer to 0 = more normal)')


---
## 6 · Multicollinearity & Feature Diagnostics

**Multicollinearity** occurs when two or more features are strongly correlated. This causes:
- Unstable coefficients in **linear models** (coefficients flip sign between runs)
- Inflated standard errors → features appear insignificant when they're not
- Redundant features waste compute and memory

### Detection: Variance Inflation Factor (VIF)

VIF measures how much one feature can be predicted by all others:

$$VIF_j = \\frac{1}{1 - R^2_j}$$

Where $R^2_j$ is the R-squared from regressing feature $j$ against all other features.

| VIF Score | Interpretation | Action |
|:-:|---|---|
| 1 | No correlation | Keep |
| 1–5 | Moderate | Monitor |
| 5–10 | High | Consider removing |
| >10 | Severe | **Remove one of the correlated pair** |


In [ ]:
# Cell 7 — Multicollinearity detection with VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

# Select numeric features for VIF analysis
vif_features = ['age', 'income', 'credit_score', 'account_age_months',
                'num_products', 'income_estimate']  # income_estimate is artificially correlated

vif_data = df_clean[vif_features].dropna()

vif_scores = pd.DataFrame()
vif_scores['Feature'] = vif_features
vif_scores['VIF'] = [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_features))]
vif_scores = vif_scores.sort_values('VIF', ascending=False)

print('=== Variance Inflation Factor (VIF) Analysis ===')
for _, row in vif_scores.iterrows():
    flag = '🔴 SEVERE' if row['VIF'] > 10 else ('🟡 HIGH' if row['VIF'] > 5 else '✅ OK')
    print(f"  {row['Feature']:25s} VIF = {row['VIF']:8.2f}  {flag}")

print(f'\n💡 Action: Drop "income_estimate" — it is a near-duplicate of "income".')
print(f'   This is common in production when two data sources provide the same metric.')

# Also show correlation matrix for the pair
corr = df_clean[['income', 'income_estimate']].corr().iloc[0, 1]
print(f'\n   Pearson correlation(income, income_estimate) = {corr:.4f}')


---
## 7 · Class Imbalance Handling

Our dataset has 80% non-churned, 20% churned. A model that predicts "no churn" for everyone gets 80% accuracy — but catches **zero** actual churners.

### Strategies Ranked by Complexity

| Strategy | How | When to Use | Caveat |
|----------|-----|-------------|--------|
| **Stratified splits** | Preserve class ratio in train/test | Always (baseline) | Doesn't fix training bias |
| **Class weights** | `class_weight='balanced'` in sklearn | Most models | May overfit minority class |
| **SMOTE** | Synthesize new minority samples in feature space | Moderate imbalance (3-10:1) | Can create noise in high dimensions |
| **Random undersampling** | Drop majority samples | Extreme imbalance + large dataset | Loses data |
| **Threshold tuning** | Adjust decision threshold from 0.5 | All classifiers | Requires calibrated probabilities |

### ⚠️ Critical Rule: SMOTE on Training Set ONLY

If you apply SMOTE before train/test split, synthetic samples based on test data leak into training. This inflates metrics and is a form of data leakage.

```
✅ Correct:  Split → SMOTE(train) → Train → Evaluate(test)
❌ Wrong:    SMOTE(all) → Split → Train → Evaluate(test)
```


In [ ]:
# Cell 8 — Class imbalance handling: SMOTE, class weights, stratified splits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import numpy as np

# Prepare features
feature_cols = ['age', 'income', 'credit_score', 'account_age_months', 'num_products',
                'has_credit_card', 'is_active', 'satisfaction_score', 'credit_score_missing']
X = df_clean[feature_cols].fillna(0)
y = df_clean['churned']

# Stratified split (preserves class ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     stratify=y, random_state=42)
print(f'Train class distribution: {dict(y_train.value_counts())}')
print(f'Test  class distribution: {dict(y_test.value_counts())}')

# === Strategy 1: Baseline (no handling) ===
lr_base = LogisticRegression(max_iter=500, random_state=42)
lr_base.fit(X_train, y_train)
print('\n--- Strategy 1: No Imbalance Handling ---')
print(classification_report(y_test, lr_base.predict(X_test), target_names=['No Churn', 'Churn']))

# === Strategy 2: Class weights ===
lr_weighted = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr_weighted.fit(X_train, y_train)
print('--- Strategy 2: class_weight="balanced" ---')
print(classification_report(y_test, lr_weighted.predict(X_test), target_names=['No Churn', 'Churn']))

# === Strategy 3: SMOTE (on training set ONLY) ===
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f'SMOTE: {len(X_train)} → {len(X_train_smote)} training samples')
print(f'  New class distribution: {dict(pd.Series(y_train_smote).value_counts())}')

lr_smote = LogisticRegression(max_iter=500, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
print('\n--- Strategy 3: SMOTE ---')
print(classification_report(y_test, lr_smote.predict(X_test), target_names=['No Churn', 'Churn']))

print('💡 Compare the RECALL for "Churn" class across all three strategies.')
print('   Higher recall = catching more actual churners (at cost of some false positives).')


---
## 8 · Production Pipeline with ColumnTransformer

In production, you **never** apply transformations manually. You build a **reproducible pipeline** that:
1. Fits on training data
2. Transforms both train and test identically
3. Can be serialized and deployed

The `ColumnTransformer` handles heterogeneous features (numeric vs. categorical) in one object.

```
ColumnTransformer
  ├── numeric:  SimpleImputer(median) → StandardScaler
  ├── categorical: SimpleImputer(most_frequent) → OneHotEncoder
  └── passthrough: binary features (already 0/1)
       |
  LogisticRegression / XGBoost / etc.
```


In [ ]:
# Cell 9 — Production sklearn Pipeline with ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Define feature groups
numeric_cols = ['age', 'income', 'credit_score', 'account_age_months', 'num_products']
categorical_cols = ['country']
binary_cols = ['has_credit_card', 'is_active', 'credit_score_missing', 'income_missing']

# Build the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore')),
        ]), categorical_cols),
        ('bin', 'passthrough', binary_cols),
    ]
)

# Full pipeline: preprocess → model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=500)),
])

# Prepare data
all_features = numeric_cols + categorical_cols + binary_cols
X = df_clean[all_features]
y = df_clean['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     stratify=y, random_state=42)

# Fit and evaluate
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('=== Production Pipeline Results ===')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Show the pipeline structure
print('Pipeline structure:')
print(f'  {pipeline}')
print(f'\n💡 This entire pipeline can be serialized with joblib and deployed.')
print(f'   It handles missing values, scaling, and encoding in one object.')
print(f'   No data leakage — .fit() on train, .transform() on test.')


---
## 9 · Automated EDA Tools

While manual EDA builds intuition, production teams use automated profiling for speed:

| Tool | Strengths | Output | Install |
|------|----------|--------|---------|
| **ydata-profiling** | Most comprehensive (correlations, interactions, alerts) | Interactive HTML report | `pip install ydata-profiling` |
| **sweetviz** | Beautiful comparison reports (train vs test) | HTML | `pip install sweetviz` |
| **D-Tale** | Interactive web UI for exploration | Browser app | `pip install dtale` |
| **DataPrep** | Fast, minimal code | HTML | `pip install dataprep` |

### When to Use Which

- **First look at new data**: `ydata-profiling` generates a full HTML report in one line
- **Comparing train vs test drift**: `sweetviz` shows distributions side-by-side
- **Exploratory deep-dive**: `dtale` gives you a spreadsheet-like interactive UI


In [ ]:
# Cell 10 — Automated EDA with ydata-profiling
# NOTE: Run this locally — it generates an interactive HTML report

# Uncomment to install and run:
# !pip install -q ydata-profiling
# from ydata_profiling import ProfileReport
# profile = ProfileReport(df_clean, title='Churn Dataset — EDA Report', explorative=True)
# profile.to_file('eda_report.html')
# print('Report saved to eda_report.html — open in browser')

# What the report contains:
print('=== What ydata-profiling Generates ===')
sections = [
    ('Overview', 'Dataset shape, types, duplicates, memory usage'),
    ('Variables', 'Distribution, stats, histogram for EVERY column'),
    ('Interactions', 'Scatter plots between all numeric pairs'),
    ('Correlations', 'Pearson, Spearman, Kendall, Phik (categorical)'),
    ('Missing Values', 'Matrix, bar chart, heatmap of missing patterns'),
    ('Alerts', '⚠️  High cardinality, high correlation, uniform, zeros'),
]
for section, desc in sections:
    print(f'  📋 {section:15s} → {desc}')

print(f'\n💡 Use this for the FIRST pass. Then do manual deep-dives on flagged issues.')


---
## 10 · Data Validation (Connects to NB07)

In production, data validation runs AUTOMATICALLY in your pipeline. Great Expectations (NB07) codifies your data assumptions as tests.

| Validation | What It Checks | Why |
|-----------|---------------|-----|
| Schema | Column names, types | Upstream schema changes break pipelines |
| Range | Min/max values | Outliers corrupt models |
| Null rate | % missing per column | Sudden null spikes = data source issue |
| Distribution | Statistical similarity to training | Drift = model degradation (NB36) |
| Uniqueness | No unexpected duplicates | ETL bugs |


In [ ]:
# Cell 11 — Lightweight data validation
def validate_dataset(df, rules):
    """Lightweight data validation (production: use Great Expectations, NB07)."""
    results = []
    for rule in rules:
        name = rule['name']
        passed = rule['check'](df)
        status = '✅ PASS' if passed else '❌ FAIL'
        results.append({'rule': name, 'status': status})
        print(f'  {status}: {name}')
    return results

rules = [
    {'name': 'No duplicate customer_ids', 'check': lambda df: df['customer_id'].nunique() == len(df)},
    {'name': 'Age between 0 and 120', 'check': lambda df: df['age'].between(0, 120).all()},
    {'name': 'Income is positive', 'check': lambda df: (df['income'] > 0).all()},
    {'name': 'Credit score 300-850', 'check': lambda df: df['credit_score'].between(300, 850).all()},
    {'name': 'No null values remaining', 'check': lambda df: df.isnull().sum().sum() == 0},
    {'name': 'Churn rate between 5-40%', 'check': lambda df: 0.05 <= df['churned'].mean() <= 0.40},
    {'name': 'At least 900 rows', 'check': lambda df: len(df) >= 900},
]

print('=== Data Validation Report (cleaned data) ===')
validate_dataset(df_clean, rules)


---
## ✅ Knowledge Check

### Q1: Imputation strategy
When should you use MEDIAN vs MEAN for imputing missing numeric values?

<details><summary>👉 Answer</summary>

Use **median** when data is skewed (income, house prices) — mean is pulled by outliers. Use **mean** when data is normally distributed (standardized test scores). In practice, median is the safer default. For production ML, use model-based imputation (KNN, iterative) for better accuracy.
</details>

### Q2: Data leakage
You're predicting customer churn. Your model has 99% accuracy. You notice `last_interaction_date` is a feature. Is this leakage?

<details><summary>👉 Answer</summary>

Potentially YES. If `last_interaction_date` was populated AFTER the churn label was determined, it leaks future information. A churned customer's last interaction is by definition before they left. The model learns "no recent interactions = churned" which is a tautology, not a prediction. Remove temporal features that are set after the prediction point.
</details>

### Q3: When NOT to scale
You're using XGBoost for classification. Should you standardize your features?

<details><summary>👉 Answer</summary>

No. Tree-based models (XGBoost, Random Forest, LightGBM) are invariant to monotonic transformations. They split on "is feature > threshold?" which doesn't change with scaling. Scaling is only needed for distance-based (KNN, SVM) and gradient-based (neural networks, logistic regression) models.
</details>

### Q4: SMOTE leakage
Your colleague applied SMOTE to the entire dataset, then did a train/test split. Why is this fundamentally wrong?

<details><summary>👉 Answer</summary>

SMOTE generates synthetic samples by interpolating between existing minority class points. If applied before splitting, some synthetic training samples are based on (or very near) actual test samples. The model effectively "memorizes" the test set through these synthetic neighbors. **Always apply SMOTE after splitting, on the training set only.**
</details>

### Q5: MNAR handling
Credit scores are missing more often for low-income customers. Is this MCAR, MAR, or MNAR? What's the best strategy?

<details><summary>👉 Answer</summary>

This is **MAR** (Missing At Random) — the missingness depends on an *observed* variable (income), not on the missing value itself. If credit scores were missing because *the credit score itself* was low (self-censoring), that would be MNAR. For MAR: use conditional imputation (impute credit_score separately for each income bracket) or model-based imputation (KNN imputer using income as a feature).
</details>


---
## 🔨 Practical Practice

### Exercise 1: Missing Data Investigation
Check if the missing values in `income` are MCAR (Missing Completely at Random) or MNAR (Missing Not at Random). Hint: compare the churn rate for rows with vs without missing income. If they differ significantly, the missingness itself is informative.

### Exercise 2: Feature Interaction
Create 3 interaction features: `income_per_product`, `age_credit_ratio`, and `engagement_score` (combine `is_active`, `satisfaction_score`, `num_products`). Do any improve churn prediction?

### Exercise 3: Full Pipeline
Build a complete sklearn Pipeline that chains: imputation → scaling → encoding → model → evaluation. Use `ColumnTransformer` for heterogeneous feature types. Test with both LogisticRegression and XGBClassifier.

### Exercise 4: VIF-Guided Feature Selection
Starting with all numeric features, iteratively remove the feature with the highest VIF until all VIF scores are below 5. Compare model performance before and after.

### Exercise 5: Automated Report
Install `ydata-profiling` and generate a full HTML report. Identify at least 3 issues that the automated tool catches that you missed in manual EDA.


---
## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Production Tool |
|---------|-----------------|----------------|
| EDA Dashboard | 6-panel diagnostic layout | matplotlib + seaborn |
| Missingness | MCAR/MAR/MNAR diagnosis & treatment | Statistical tests + indicators |
| Cleaning | Order-dependent pipeline | pandas |
| Encoding | One-hot, target, ordinal, frequency | sklearn encoders |
| Scaling | Standard, Robust, Log transforms | sklearn scalers |
| Multicollinearity | VIF analysis & feature removal | statsmodels |
| Class Imbalance | SMOTE, class weights, threshold tuning | imbalanced-learn |
| Production Pipeline | ColumnTransformer + Pipeline | scikit-learn |
| Automated EDA | One-line profiling reports | ydata-profiling |
| Validation | Rule-based data checks | Great Expectations (NB07) |

### The 5 Rules of Data Preprocessing

1. **Always diagnose before imputing** — MCAR, MAR, and MNAR require different strategies
2. **Clean outliers before computing statistics** — one impossible value corrupts all downstream steps
3. **SMOTE on training set ONLY** — never before the train/test split
4. **Use pipelines, not manual steps** — reproducibility prevents production bugs
5. **Tree models don't need scaling** — but linear models and neural networks do

---

**Connections:** Data validation concepts → NB07 (Great Expectations) | Feature engineering → NB11 (Feast Feature Store) | Visual analysis → NB36 (EvidentlyAI Drift Detection) | Full ML pipeline → NB13_02 (Linear Models) | Class imbalance → NB13_07 (Model Evaluation)
